### Exercise 1

#### Part 1

Below I use the D2L lecture-note setting: grayscale input $1 \times 224 \times 224$, VGG-11 architecture
$$
((1,64),(1,128),(2,256),(2,512),(2,512)),
$$
and 10 output classes. The note says VGG keeps the AlexNet-style fully connected head but replaces the individually designed conv layers with repeated VGG blocks.

AlexNet

Using the D2L AlexNet-style architecture:
Conv layers: $1 \rightarrow 96 \rightarrow 256 \rightarrow 384 \rightarrow 384 \rightarrow 256$
and then
$$
256 \cdot 5 \cdot 5=6400
$$
flattened features going into the fully connected layers.

| Part | Parameters |
| :--- | :--- |
| Convolutional layers | 3.72 million |
| Fully connected layers | 43.04 million |
| Total | 46.76 million |

So most AlexNet parameters come from the fully connected layers.

VGG-11

For VGG-11, after five $2 \times 2$ max-pooling layers,
$$
224 \rightarrow 112 \rightarrow 56 \rightarrow 28 \rightarrow 14 \rightarrow 7 .
$$
So before the classifier, the feature map is
$$
512 \times 7 \times 7=25088 .
$$
Then the fully connected head is
$$
25088 \rightarrow 4096 \rightarrow 4096 \rightarrow 10 .
$$
| Part | Parameters |
| :--- | :--- |
| Convolutional layers | 9.22 million |
| Fully connected layers | 119.59 million |
| Total | 128.81 million |

So VGG-11 has about
$$
\frac{128.81}{46.76} \approx 2.75
$$
times as many parameters as AlexNet.

The biggest reason is the first fully connected layer:
$$
25088 \times 4096 \approx 102.76 \text { million }
$$
parameters by itself.

#### Part 2

I will count MACs, multiply-accumulate operations. If your class counts one multiply and one addition as two FLOPs, just multiply the numbers below by 2 .

For a convolutional layer, the approximate MAC count is
$$
H_{\text {out }} W_{\text {out }} C_{\text {out }} C_{\text {in }} K^2 .
$$
For a fully connected layer, it is
$$
d_{\mathrm{in}} d_{\mathrm{out}}
$$
| Part | MACs |
| :--- | :--- |
| Convolutional layers | 895.11 million |
| Fully connected layers | 43.03 million |
| Total | 938.15 million |

Even though the fully connected layers dominate the parameter count, the convolutional layers dominate the computation.

VGG-11

| Part | MACs |
| :--- | :--- |
| Convolutional layers | 7.43 billion |
| Fully connected layers | 119.58 million |
| Total | 7.55 billion |

So VGG-11 uses about
$$
\frac{7.55}{0.938} \approx 8.0
$$
times as many MACs as AlexNet.

The main increase comes from the convolutional layers:
$$
\frac{7.43 \mathrm{~B}}{0.895 \mathrm{~B}} \approx 8.3 .
$$
Intuitively, VGG does many more $3 \times 3$ convolutions at relatively high spatial resolutions. For example, early VGG layers operate on $224 \times 224,112 \times 112$, and $56 \times 56$ feature maps, which is expensive.

#### Part 3

**Use $1 \times 1$ convolutions before flattening**

A $1 \times 1$ convolution can reduce the channel dimension:
$$
512 \times 7 \times 7 \rightarrow 128 \times 7 \times 7 .
$$
Then the flattened vector becomes
$$
128 \times 7 \times 7=6272,
$$
which is much smaller than 25088 .

### Exercise 3

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def vgg_block(num_convs, out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.LazyConv2d(out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers)

In [3]:
class VGG(d2l.Classifier):
    def __init__(self, arch, lr=0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        conv_blks = []
        for (num_convs, out_channels) in arch:
            conv_blks.append(vgg_block(num_convs, out_channels))
        self.net = nn.Sequential(
            *conv_blks, nn.Flatten(),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(num_classes))
        self.net.apply(d2l.init_cnn)

In [5]:
vgg16_arch = (
    (2, 64),
    (2, 128),
    (3, 256),
    (3, 512),
    (3, 512)
)

model_vgg16 = VGG(
    arch = vgg16_arch,
    lr=0.01,
    num_classes=10
)

model_vgg16.net

/Users/zouminghao/Desktop/d2l-notes-exercises/venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


Sequential(
  (0): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (2): Sequential(
    (0): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (3

In [6]:
vgg19_arch = (
    (2, 64),
    (2, 128),
    (4, 256),
    (4, 512),
    (4, 512)
)

model_vgg19 = VGG(
    arch = vgg19_arch,
    lr=0.01,
    num_classes=10
)

model_vgg19.net

/Users/zouminghao/Desktop/d2l-notes-exercises/venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


Sequential(
  (0): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (2): Sequential(
    (0): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU

### Exercise 4